In [ ]:
!pip install -q "trl[peft]>=0.21.0" "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 11.2 MB/s eta 0:00:00


In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.5 MB/s eta 0:00:00


#OLMo Baseline on GSM8K

In [ ]:
import re, math, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "allenai/OLMo-2-0425-1B"

ds = load_dataset("gsm8k", "main")
test_ds = ds["test"]   # 1319

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("Loaded:", MODEL_NAME, " | test size:", len(test_ds))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Loaded: allenai/OLMo-2-0425-1B  | test size: 1319


In [ ]:
def build_prompt(question: str) -> str:
    return (
        "You are a helpful assistant that solves math word problems.\n"
        "Solve the problem step by step, then give the final answer as a number.\n\n"
        f"Question: {question}\n"
        "Answer: Let's think step by step.\n"
    )

_gold_re = re.compile(r"####\s*([-+]?\d+(?:\.\d+)?)")

def extract_gold(answer_text: str):
    # Model Answer
    m = _gold_re.search(answer_text)
    if not m:
        return None
    return m.group(1)

def extract_pred(text: str):
    # Model Digital Answer
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    if not nums:
        return None
    return nums[-1]

def normalize_num_str(x: str):
    # Normalize integer and float number
    if x is None:
        return None
    try:
        v = float(x)
        if abs(v - round(v)) < 1e-9:
            return str(int(round(v)))
        return str(v)
    except:
        return None

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def evaluate_gsm8k(max_examples=None, max_new_tokens=256):
    n = len(test_ds) if max_examples is None else min(max_examples, len(test_ds))
    correct = 0
    total = 0

    for i in tqdm(range(n)):
        ex = test_ds[i]
        q = ex["question"]
        gold = normalize_num_str(extract_gold(ex["answer"]))

        prompt = build_prompt(q)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        pred = normalize_num_str(extract_pred(gen))

        is_correct = (pred is not None) and (gold is not None) and (pred == gold)
        correct += int(is_correct)
        total += 1

        # if i < 3:
        #     print("Q:", q)
        #     print("GOLD:", gold)
        #     print("PRED:", pred)
        #     print("GEN:", gen[:400], "...\n")

    acc = correct / max(total, 1)
    return {"accuracy": acc, "correct": correct, "total": total}

res_small = evaluate_gsm8k(max_examples=50, max_new_tokens=256)
print(res_small)



  2%|▏         | 1/50 [00:04<03:46,  4.62s/it]

Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
GOLD: 18
PRED: 26
GEN: Step 1: Janet eats 3 eggs for breakfast every morning.
Step 2: She sells the remainder at the farmers' market daily for $2 per fresh duck egg.
Step 3: She sells 16 - 3 = 13 eggs at the farmers' market.
Step 4: She sells 13 eggs at $2 each, so she makes 13 * $2 = $26 every day at the farmers' market.
Step 5: She makes $26 every day at the farmers' market.
Answer: Janet makes $26 every day at the fa ...



  4%|▍         | 2/50 [00:06<02:11,  2.73s/it]

Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
GOLD: 3
PRED: 3
GEN: The robe takes 2 bolts of blue fiber and half that much white fiber.
So, the robe takes 2 / 2 = 1 bolt of white fiber.
The total number of bolts of fiber is 2 + 1 = 3 bolts.
Answer: 3 ...



  6%|▌         | 3/50 [00:08<01:58,  2.53s/it]

Q: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
GOLD: 70000
PRED: 50000
GEN: The house was initially worth $80,000. After repairs, the value increased by 150%. So, the value of the house after repairs is $80,000 + $50,000 = $130,000.
The profit is the difference between the final value and the initial value. So, the profit is $130,000 - $80,000 = $50,000.
The final answer: 50000. ...



100%|██████████| 50/50 [03:46<00:00,  4.53s/it]

{'accuracy': 0.34, 'correct': 17, 'total': 50}


In [ ]:
res_full = evaluate_gsm8k(max_examples=None, max_new_tokens=256)
print(res_full)

  0%|          | 1/1319 [00:03<1:06:17,  3.02s/it]

Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
GOLD: 18
PRED: 26
GEN: Step 1: Janet eats 3 eggs for breakfast every morning.
Step 2: She sells the remainder at the farmers' market daily for $2 per fresh duck egg.
Step 3: She sells 16 - 3 = 13 eggs at the farmers' market.
Step 4: She sells 13 eggs at $2 each, so she makes 13 * $2 = $26 every day at the farmers' market.
Step 5: She makes $26 every day at the farmers' market.
Answer: Janet makes $26 every day at the fa ...



  0%|          | 2/1319 [00:04<45:23,  2.07s/it]  

Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
GOLD: 3
PRED: 3
GEN: The robe takes 2 bolts of blue fiber and half that much white fiber.
So, the robe takes 2 / 2 = 1 bolt of white fiber.
The total number of bolts of fiber is 2 + 1 = 3 bolts.
Answer: 3 ...



  0%|          | 3/1319 [00:06<46:27,  2.12s/it]

Q: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
GOLD: 70000
PRED: 50000
GEN: The house was initially worth $80,000. After repairs, the value increased by 150%. So, the value of the house after repairs is $80,000 + $50,000 = $130,000.
The profit is the difference between the final value and the initial value. So, the profit is $130,000 - $80,000 = $50,000.
The final answer: 50000. ...



100%|██████████| 1319/1319 [1:37:13<00:00,  4.42s/it]

{'accuracy': 0.3737680060652009, 'correct': 493, 'total': 1319}


#OLMo Baseline on AIME

In [ ]:
from datasets import load_dataset

ds_aime = load_dataset("MathArena/aime_2025")
aime_test = ds_aime["train"]
aime_test

Dataset({
    features: ['problem_idx', 'problem', 'answer', 'problem_type'],
    num_rows: 30
})

In [ ]:
from tqdm import tqdm
import torch

def build_aime_prompt(problem: str) -> str:
    return (
        "Solve the following competition math problem. "
        "Give only the final answer as a 3-digit integer from 000 to 999.\n\n"
        f"Problem: {problem}\n"
        "Answer: "
    )

def normalize_aime_answer(x):

    if x is None:
        return None
    s = str(x).strip()

    try:
        v = int(s)
    except:
        return None
    if v < 0 or v > 999:
        return None
    return f"{v:03d}"

def extract_aime_pred(text: str):

    nums = re.findall(r"\b\d+\b", text)
    if not nums:
        return None

    for n in reversed(nums):
        norm = normalize_aime_answer(n)
        if norm is not None:
            return norm
    return None

@torch.no_grad()
def evaluate_aime_batched(dataset, batch_size=8, max_examples=None, max_new_tokens=256):
    n = len(dataset) if max_examples is None else min(max_examples, len(dataset))
    correct = 0
    total = 0

    for start in tqdm(range(0, n, batch_size)):
        batch = dataset.select(range(start, min(start + batch_size, n)))

        prompts = [build_aime_prompt(ex["problem"]) for ex in batch]
        golds   = [normalize_aime_answer(ex["answer"]) for ex in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        prompt_lens = inputs["attention_mask"].sum(dim=1).tolist()

        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        for i in range(out.size(0)):
            gen_ids = out[i, prompt_lens[i]:]
            gen_txt = tokenizer.decode(gen_ids, skip_special_tokens=True)

            pred = extract_aime_pred(gen_txt)
            gold = golds[i]

            is_correct = (pred is not None) and (gold is not None) and (pred == gold)
            correct += int(is_correct)
            total += 1

            if total <= 3:
                print("\n--- SAMPLE", total, "---")
                print("GOLD:", gold, " raw:", batch[i]["answer"])
                print("PRED:", pred)
                print("GEN :", gen_txt[:250], "...")

    return {"accuracy": correct / max(total, 1), "correct": correct, "total": total}

In [ ]:
evaluate_aime_batched(aime_test, max_examples=10)

 50%|█████     | 1/2 [00:07<00:07,  7.19s/it]


--- SAMPLE 1 ---
GOLD: 070  raw: 70
PRED: 005
GEN :  the sum of 3^2$ is 3

$\sum of the sum of 3

$\sum of 3

$\sum of all the numbers of the sum of 3

Problem: 3

Problem: Find the sum of all the numbers of the sum of all integers from 1 to 1000 that are divisible by 3 and 5.

Problem: Find the sum o ...

--- SAMPLE 2 ---
GOLD: 588  raw: 588
PRED: 288
GEN : 288

\end{document}

enter image description here

I'm not sure how to get the area of the heptagon. I tried using the area of the triangle and the area of the quadrilateral, but I don't know how to get the area of the heptagon. I also tried using th ...

--- SAMPLE 3 ---
GOLD: 016  raw: 16
PRED: 003
GEN :  1000 is the number of players. 3 is 1000 is 3-digit 1000 is 3 is 1000 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 is 3 ...


100%|██████████| 2/2 [00:14<00:00,  7.09s/it]


{'accuracy': 0.0, 'correct': 0, 'total': 10}